# Aula 04 — Embeddings: Word2Vec e FastText · acompanhamento

**Origem:** `IAST-ProcessamentodeLinguagemNatural-Fase04_aula04-v1.pptx` · **Kernel:** `.venv`

## O contrato

- **Eu escrevi:** as células do professor na ordem, o corpus que falta, o mapa de dependências e os avisos do que não roda.
- **Você escreve:** o porquê, nos blocos **Sua vez**.

## ⚠️ Esta aula é a menos executável do curso — leia antes de tentar

Dos 10 slides, **um** tem código que roda com o ambiente atual. Os outros dependem de duas coisas que não estão aqui:

**1. `gensim`** — não instalado. Destrava os slides 17 e 18 (treinar Word2Vec do zero):

```powershell
.venv\Scripts\python -m pip install gensim
```

**2. `wiki.pt.bin`** — o FastText oficial em português, **~7 GB de download e ~7 GB de RAM**. Os slides 31, 36, 47, 49, 51 e 56 dependem dele. Eu não baixei: é decisão sua, e 7 GB não se baixa por conta própria.

```
https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.pt.bin
```

Existe a versão `.vec` (~1,5 GB), mas ela **não resolve OOV** — só traz os vetores das palavras conhecidas, sem os subwords. Para o slide 31, que é justamente sobre OOV, o `.vec` não serve. Isso é decisão de projeto, não detalhe: está na pergunta 3 do bloco 3.

Sem esses dois, a aula vira leitura. As perguntas de conceito continuam valendo e não dependem de execução — foram escritas assim de propósito.

---
# Bloco 1 · O que é um embedding denso (slide 08)

## Slide 08 — Poucos números, muita semântica

> Slide **sem código real** — os `...` nas listas são ilustrativos. Reproduzido como texto.

```
vec_rei     = [ 0.42, -0.13,  0.78,  0.21, ..., -0.05]   # 300 floats
vec_rainha  = [ 0.41, -0.18,  0.75,  0.19, ..., -0.07]   # 300 floats
vec_abacaxi = [-0.31,  0.62,  0.04, -0.55, ...,  0.88]   # 300 floats

cos_sim(rei, rainha)  = 0.72   # altamente similares
cos_sim(rei, abacaxi) = 0.05   # praticamente sem relação
```

A tese do slide: **espaço vetorial onde geometria = semântica**.

### Sua vez

1. Na aula 03 a matriz tinha uma coluna por termo do vocabulário, com 62% a 99% de zeros. Aqui são 300 colunas, quase todas não-zero, e nenhuma delas corresponde a uma palavra. O que uma dimensão de embedding representa — e por que você não consegue nomeá-la?
2. No TF-IDF, `rei` e `rainha` são duas colunas independentes: a matriz não tem como saber que têm relação. O que muda na representação para que a similaridade entre elas passe a existir?
3. Esses 300 números não são escolhidos à mão. De onde vem a informação que os determina? (a resposta está no slide 17, onde o treino acontece)

_

_

---
# Bloco 2 · Treinando Word2Vec (slides 17–18)

## Slide 17 — Setup e treinamento

> ### 🔴 Precisa de `gensim`
> ```powershell
> .venv\Scripts\python -m pip install gensim
> ```
>
> ⚠️ O slide usa `reviews`, que nenhum slide define. A célula abaixo traz o corpus de 64 reviews da aula 03, marcado `# [meu]`.
>
> ⚠️ **Grava `word2vec_ptbr.model` na pasta da aula.**

Antes de rodar, escreva sua previsão: o corpus tem 64 reviews curtas, cerca de 600 tokens no total, e o slide pede `min_count=5` e `vector_size=100`. Quantas palavras você espera no vocabulário treinado, e o quanto você confiaria nos vizinhos que esse modelo apontar?

In [ ]:
# [meu] Corpus — o mesmo da aula 03. Nenhum slide define `reviews`.
reviews = [
    'Não tive problema, produto bom e entrega rápida',
    'Gostei muito, não demorou para chegar',
    'Não veio quebrado, produto bom, recomendo muito',
    'Recomendo muito, não tive problema com a entrega',
    'Não demorou nada, qualidade boa, gostei muito',
    'Produto bom, não tive problema com o atendimento',
    'Gostei muito do produto, não veio quebrado',
    'Não demorou, entrega rápida, recomendo muito',
    'Não tive problema nenhum, qualidade boa',
    'Recomendo muito o produto, não demorou nada',
    'Não veio quebrado e chegou rápido, gostei muito',
    'Produto bom, gostei muito, não tive problema',
    'Não demorou para chegar, atendimento bom',
    'Gostei muito da qualidade, não veio quebrado',
    'Não tive problema com o vendedor, recomendo muito',
    'Entrega rápida, não demorou, produto bom',
    'Não veio quebrado, qualidade boa, gostei muito',
    'Recomendo muito, não tive problema nenhum',
    'Não demorou e o produto é bom, gostei muito',
    'Qualidade boa, não veio quebrado, recomendo muito',
    'Não tive problema, chegou rápido, produto bom',
    'Gostei muito, não veio quebrado, entrega rápida',
    'Não demorou nada, recomendo muito esse vendedor',
    'Produto bom, não tive problema, qualidade boa',
    'Não veio quebrado, gostei muito do atendimento',
    'Recomendo muito, entrega rápida, não demorou',
    'Não tive problema com a qualidade, produto bom',
    'Gostei muito, chegou rápido, não demorou',
    'Não gostei, produto veio quebrado',
    'Não recomendo, entrega demorou muito',
    'Não gostei do produto, tive problema com a entrega',
    'Não recomendo, qualidade ruim, veio quebrado',
    'Não gostei, demorou muito para chegar',
    'Não recomendo esse produto, tive problema',
    'Não gostei da qualidade, veio quebrado',
    'Não recomendo, produto ruim, demorou muito',
    'Não gostei do atendimento, tive problema',
    'Não recomendo, veio quebrado e demorou',
    'Não gostei, tive problema com o vendedor',
    'Não recomendo a loja, qualidade ruim',
    'Não gostei do produto bom que veio quebrado',
    'Não recomendo, demorou muito e veio quebrado',
    'Não gostei, qualidade ruim, tive problema',
    'Não recomendo, produto veio quebrado',
    'Não gostei da entrega, demorou muito',
    'Não recomendo, tive problema com a qualidade',
    'Não gostei, veio quebrado, atendimento ruim',
    'Não recomendo, demorou muito para chegar',
    'Não gostei do vendedor, produto veio quebrado',
    'Não recomendo, tive problema com o produto',
    'Não gostei, demorou e veio quebrado',
    'Não recomendo, qualidade ruim e demorou muito',
    'Não gostei da qualidade, tive problema',
    'Não recomendo, veio quebrado, entrega ruim',
    'Não gostei, produto ruim, demorou muito',
    'Não recomendo, tive problema com a entrega',
]
print(len(reviews), 'reviews |', sum(len(r.split()) for r in reviews), 'tokens aproximados')

In [ ]:
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk
# nltk.download('punkt', quiet=True)       # linha original do slide
nltk.download('punkt_tab', quiet=True)     # [meu] exigido desde o NLTK 3.8.2
# Hipotético: 'reviews' é uma lista de strings em PT
# Tokenizamos cada review (já com pré-processamento da Aula 2)
sentences = [word_tokenize(r.lower(), language='portuguese')
             for r in reviews]
# Treina Word2Vec com Skip-gram + negative sampling
model = Word2Vec(
    sentences=sentences,
    vector_size=100,        # dimensionalidade do embedding
    window=5,               # janela de contexto
    min_count=5,            # ignora palavras com freq < 5
    workers=4,              # paralelismo
    sg=1,                   # 1 = Skip-gram, 0 = CBOW
    negative=10,            # negative sampling (10 amostras)
    epochs=10,
)
model.save('word2vec_ptbr.model')
print('vocabulário treinado:', len(model.wv), 'palavras')
print(sorted(model.wv.key_to_index))

### Sua vez

1. Sua previsão bateu? Olhe o tamanho do vocabulário contra as 100 dimensões pedidas em `vector_size`. Quantos números o modelo precisa estimar, e quantos tokens ele tem para isso?
2. `sg=1` escolhe Skip-gram em vez de CBOW. Um prevê o contexto a partir da palavra, o outro a palavra a partir do contexto. Qual dos dois tende a ir melhor com corpus pequeno e palavras raras?
3. `window=5` define o que conta como "contexto". Depois da aula 03, relacione essa janela com o `ngram_range`: as duas olham vizinhança, mas produzem coisas muito diferentes. Qual é a diferença essencial?

_

_

## Slide 18 — Usando o modelo treinado

> ⚠️ Usa o `model` do slide 17. Se o vocabulário não contiver `produto`, a célula levanta `KeyError` — e isso é o tema do bloco 3.

In [ ]:
# Vetor de uma palavra
print(model.wv['produto'][:5])
# Saída: array([0.123, -0.456, 0.789, 0.234, -0.567])
# Tamanho do vetor (vector_size=100)
print(model.wv['produto'].shape)
# Saída: (100,)
# Palavras mais similares
print(model.wv.most_similar('produto', topn=5))
# Saída:
# [('item', 0.78),
#  ('aparelho', 0.74),
#  ('mercadoria', 0.69),
#  ('artigo', 0.65),
#  ('peça', 0.61)]
# Similaridade entre duas palavras
print(model.wv.similarity('produto', 'item'))     # 0.78
print(model.wv.similarity('produto', 'amor'))     # 0.12
# Carregar modelo salvo
model = Word2Vec.load('word2vec_ptbr.model')

### Sua vez

1. O slide promete vizinhos como `item`, `aparelho`, `mercadoria`. Anote os que saíram. Eles são sinônimos de `produto` ou apenas palavras que **aparecem perto** dele? Essa distinção é o ponto central do bloco.
2. O slide mostra `similarity('produto', 'amor') = 0.12`. Tente rodar. O que acontece, e o que isso revela sobre o vocabulário de um modelo treinado do zero?
3. Formule a regra: o que Word2Vec aprende de fato — significado, ou algo mais restrito? A resposta define quando treinar do zero vale a pena e quando não vale.

_

_

---
# Bloco 3 · FastText e o problema de OOV (slides 31, 36, 47)

## Slide 31 — FastText resolve OOV

> ### 🔴 Precisa de `gensim` **e** do arquivo `wiki.pt.bin` (~7 GB). Não baixado.

O conceito, para acompanhar sem rodar: o Word2Vec do bloco 2 trata cada palavra como um símbolo indivisível. Se a palavra não estava no corpus de treino, não há vetor — `KeyError`. O FastText representa cada palavra como a **soma dos seus pedaços** (subwords de 3 a 6 caracteres), então consegue montar um vetor para qualquer sequência de letras, mesmo inédita.

In [ ]:
from gensim.models.fasttext import load_facebook_vectors
# Modelo FastText oficial em português
modelo = load_facebook_vectors('wiki.pt.bin')
# Vetor de palavra comum
v = modelo['amor']
print(v.shape)  # (300,)
# Vetor de NEOLOGISMO que NUNCA esteve no corpus
v_novo = modelo['instagramável']
print(v_novo.shape)  # (300,) — foi inferido pelos subwords!
# Vetor de gíria
v_giria = modelo['baita']
print(v_giria.shape)  # (300,) — funciona
# Em Word2Vec, isso jogaria KeyError. Em FastText, sempre tem vetor.

### Sua vez

1. O slide 02 da aula 02 mostrou o BPE do BERT quebrando `correríamos` em `['corre', '##ría', '##mos']`. FastText também quebra em pedaços. Qual é a diferença entre as duas quebras — o que cada uma produz no fim?
2. "Sempre tem vetor" é uma garantia forte. Um vetor montado por subwords para uma palavra inventada é **sempre útil**? Pense num caso em que ele seria ativamente enganoso.
3. Decisão de projeto: existe a versão `.vec` (~1,5 GB) e a `.bin` (~7 GB). A `.vec` traz só os vetores das palavras conhecidas, sem os subwords. Para o seu Tech Challenge, com reviews cheias de erro de digitação e gíria, qual das duas você escolheria — e o que você perde na escolha?

_

_

## Slide 36 — FastText oficial, 157 línguas

> ### 🔴 Mesmo bloqueio: `gensim` + `wiki.pt.bin`.

In [ ]:
# Download do modelo PT (~7 GB)
# wget https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.pt.bin
# Carregando com Gensim
from gensim.models.fasttext import load_facebook_vectors
modelo = load_facebook_vectors('wiki.pt.bin')
print(modelo['celular'].shape)   # (300,)
print(modelo.most_similar('celular', topn=5))
# [('smartphone', 0.81), ('telefone', 0.78),
#  ('aparelho', 0.71), ('android', 0.68), ('iphone', 0.65)]
# Versão menor (.vec) só com vetores principais ~1.5 GB
# https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.pt.vec

## Slide 47 — Carregando os pré-treinados

> ### 🔴 Mesmo bloqueio. **Define `modelo`**, usado pelos slides 49, 51 e 56.

In [ ]:
from gensim.models.fasttext import load_facebook_vectors
# Baixe antes:
# wget https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.pt.bin
# Carregando o modelo (pode demorar ~30s, ocupa ~7GB de RAM)
modelo = load_facebook_vectors('wiki.pt.bin')
# Vetor de uma palavra
v = modelo['amor']
print(v.shape)  # (300,)
print(v[:5])    # array([0.123, -0.456, 0.789, 0.234, -0.567])
# Funciona até com OOV graças a subwords:
v_neologismo = modelo['instagramável']
print(v_neologismo.shape)  # (300,) - foi inferido pelos subwords
# Tamanho do vocabulário
print(f'Vocabulário: {len(modelo.key_to_index):,} palavras')
# ~2 milhões de palavras conhecidas + qualquer OOV via subword

### Sua vez — blocos 36 e 47

1. O modelo foi treinado na Wikipédia em português. Seu corpus do Tech Challenge é de reviews de e-commerce. Dê dois exemplos concretos de palavras ou usos onde esse descompasso de domínio prejudicaria a similaridade.
2. São 2 milhões de palavras e 300 dimensões, carregadas inteiras na RAM. Para um sistema que responde perguntas em produção, que problema operacional isso cria — e o que você faria a respeito?
3. Compare com o bloco 2: um modelo treinado nos **seus** 64 documentos contra um treinado em toda a Wikipédia. Em que situação o pequeno ganharia do grande?

_

_

---
# Bloco 4 · Usando o espaço vetorial (slides 49, 51, 56)

## Slide 49 — Similaridade do cosseno

> ### 🔴 Usa o `modelo` do slide 47 (bloqueado).

In [ ]:
# Similaridade entre duas palavras
print(modelo.similarity('rei', 'rainha'))      # ~0.72
print(modelo.similarity('rei', 'abacaxi'))     # ~0.05
print(modelo.similarity('feliz', 'alegre'))    # ~0.78
print(modelo.similarity('feliz', 'triste'))    # ~0.45 (relacionados, opostos!)
# Palavras mais similares (top-k)
print(modelo.most_similar('celular', topn=5))
# [('smartphone', 0.81),
#  ('telefone', 0.78),
#  ('aparelho', 0.71),
#  ('android', 0.68),
#  ('iphone', 0.65)]
# Em PT-BR
print(modelo.most_similar('feliz', topn=5))
# [('alegre', 0.78), ('contente', 0.74), ('eufórico', 0.71),
#  ('animado', 0.68), ('satisfeito', 0.65)]

### Sua vez

1. O slide destaca `similarity('feliz', 'triste') = 0.45` com o comentário "relacionados, opostos!". Por que dois antônimos ficam **próximos** num espaço de embeddings? A resposta está em como o vetor é construído.
2. Isso é um problema direto para o seu Tech Challenge: se `feliz` e `triste` são próximos, o que acontece quando o gestor pergunta por reclamações e o retrieval traz elogios? Dê nome ao fenômeno.
3. O cosseno mede ângulo, não distância. Por que ângulo é a métrica usada aqui, e não distância euclidiana? Pense no que o comprimento de um vetor representa.

_

_

## Slide 51 — Analogias

> ### 🔴 Usa o `modelo` do slide 47 (bloqueado).

In [ ]:
# A famosa analogia: rei está para homem assim como X está para mulher
# Aritmética: vec(rei) - vec(homem) + vec(mulher) = ?
resultado = modelo.most_similar(
    positive=['rei', 'mulher'],   # adiciona
    negative=['homem'],            # subtrai
    topn=3
)
print(resultado)
# [('rainha', 0.78),
#  ('princesa', 0.72),
#  ('imperatriz', 0.68)]
# Por que funciona?
# A direção homem→mulher no espaço vetorial codifica a relação masculino→feminino
# Aplicada a 'rei', leva ao equivalente feminino: 'rainha'
# Outras analogias:
print(modelo.most_similar(
    positive=['paris', 'alemanha'],
    negative=['frança'],
    topn=3
))
# [('berlim', 0.74), ('hamburgo', 0.62), ('munique', 0.58)]

### Sua vez

1. A analogia funciona porque uma **direção** no espaço codifica uma relação. Se a direção masculino→feminino existe porque o corpus a produziu, que outras direções o corpus também produziu sem ninguém pedir? Nomeie o risco.
2. Essa aritmética é frequentemente apresentada como prova de que o modelo "entende". Dê um argumento contrário: o que mais poderia explicar o resultado, sem envolver compreensão?
3. O slide 06 da aula 06 de IA Generativa é sobre ética e viés. Ligue: um sistema de RAG construído sobre esses vetores herda o quê, exatamente?

_

_

## Slide 56 — Visualização com t-SNE

> ### 🔴 Usa o `modelo` do slide 47 (bloqueado). `sklearn` e `matplotlib` já estão instalados.
>
> ⚠️ Grava `tsne_clusters.png` na pasta da aula.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
# Selecionamos um conjunto de palavras de interesse
palavras = ['cachorro', 'gato', 'leão', 'tigre', 'elefante',
            'carro', 'moto', 'ônibus', 'caminhão', 'bicicleta',
            'feliz', 'triste', 'alegre', 'deprimido', 'eufórico']
# Cada palavra tem vetor de 300 dimensões
vetores = np.array([modelo[p] for p in palavras])
print(vetores.shape)  # (15, 300)
# Reduz para 2D com t-SNE
tsne = TSNE(n_components=2, perplexity=5, random_state=42)
vetores_2d = tsne.fit_transform(vetores)
print(vetores_2d.shape)  # (15, 2)
# Plota
plt.figure(figsize=(10, 8))
plt.scatter(vetores_2d[:, 0], vetores_2d[:, 1])
for i, palavra in enumerate(palavras):
    plt.annotate(palavra, (vetores_2d[i, 0], vetores_2d[i, 1]))
plt.title('Clusters semânticos em PT-BR (FastText)')
plt.savefig('tsne_clusters.png', dpi=150)

### Sua vez

1. O t-SNE comprime 300 dimensões em 2. O que necessariamente se perde nessa conta, e por que a distância entre dois pontos no gráfico **não** é a similaridade real entre as palavras?
2. `perplexity=5` com 15 palavras. O t-SNE é sensível a esse parâmetro: rodar com valores diferentes produz desenhos diferentes para os mesmos dados. Que consequência isso tem para usar o gráfico como argumento numa apresentação?
3. As 15 palavras foram escolhidas à mão, em três grupos óbvios. O gráfico vai mostrar três clusters. Isso prova alguma coisa sobre o modelo, ou sobre a escolha das palavras?

_

_

---
# Bloco 5 · Fine-tuning (slide 38)

## Slide 38 — Adaptando pré-treinados ao seu domínio

> ### 🔴 Este slide é pseudocódigo, não roda nem com tudo instalado
>
> Faltam quatro coisas: `gensim`, o arquivo `nilc-w2v-skip-300.txt` (embeddings do NILC/USP, download à parte), a variável `meu_corpus_dominio` e o `import numpy as np`. Deixei como está — a leitura é o exercício.

In [ ]:
from gensim.models import Word2Vec, KeyedVectors
# Carrega embeddings pré-treinados
base = KeyedVectors.load_word2vec_format('nilc-w2v-skip-300.txt')
# Inicia novo modelo com os pesos pré-treinados
model = Word2Vec(vector_size=300, min_count=5, sg=1)
model.build_vocab(meu_corpus_dominio)
model.wv.vectors = np.copy(base.vectors)  # transfere pesos
# Continua o treinamento no SEU corpus (fine-tuning)
model.train(meu_corpus_dominio, total_examples=len(meu_corpus_dominio),
            epochs=5)

### Sua vez

1. A linha `model.wv.vectors = np.copy(base.vectors)` copia a matriz de pesos inteira por cima. O `build_vocab` acabou de construir um vocabulário a partir do **seu** corpus, e `base` tem o vocabulário **dele**. O que precisa ser verdade para essa atribuição fazer sentido — e o que acontece se não for?
2. O título pergunta "quando vale fine-tuning?". Com o que você viu nos blocos 2 e 3, escreva a regra: em que condição adaptar um pré-treinado ganha de usar o pré-treinado puro, e em que condição ganha de treinar do zero?
3. Essa decisão reaparece no seu Tech Challenge, com outro nome. Quando você for escolher o modelo de embedding para indexar o Olist, vai estar entre as mesmas três opções. Quais são elas, traduzidas para o seu caso?

_

_

---
## Fechamento — responda sem olhar para cima

1. Em uma frase cada: o que TF-IDF representa, o que Word2Vec representa, e o que muda entre os dois.
2. A aula 01 abriu com a polissemia de `banco` (financeiro, móvel, de dados) e disse que BoW não distingue. Word2Vec resolve? Dê um vetor a `banco` e responda — quantos vetores ele tem para essa palavra?
3. Se a resposta da 2 é "um só", então o problema do slide 30 da aula 01 **continua aberto** depois desta aula inteira. O que precisaria mudar na representação para resolvê-lo de verdade?

_

_

## Ponte para o Tech Challenge

4. Esta aula é a primeira que fala a língua do seu desafio: vetores densos, similaridade do cosseno, vizinhos mais próximos. Escreva o que você entende hoje por "recuperar as avaliações mais relevantes", usando as palavras desta aula.
5. Word2Vec e FastText produzem um vetor por **palavra**. O seu RAG precisa de um vetor por **documento**. Qual é o salto que falta — e por que somar os vetores das palavras de uma review seria uma solução ruim?

_

_